In [1]:
!pip install -q torch "transformers==4.44.2" accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.9 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, set_seed

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
model_id = "microsoft/Phi-3.5-mini-instruct"

print(f"Downloading and loading {model_id}...")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load Model
# We skip complex GPU optimizations but use float16 for memory efficiency on Colab T4 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print("Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Model loaded successfully.


In [4]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def generate_response(prompt, max_new_tokens=500):
    """
    Helper function to reproduce the inference shown in the video.
    It applies the chat template and generates text.
    """
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # Generate output
    output = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        return_full_text=False,
        temperature=0.0, # Making it deterministic as implied by set_seed
        do_sample=False,
    )
    return output[0]['generated_text']

In [5]:
prompt = "What about solving an 2x + 3 = 7 equation?"
print(generate_response(prompt))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


 To solve the linear equation 2x + 3 = 7, follow these steps:

Step 1: Isolate the term with the variable (2x) by subtracting 3 from both sides of the equation.

2x + 3 - 3 = 7 - 3

This simplifies to:

2x = 4

Step 2: Solve for the variable (x) by dividing both sides of the equation by the coefficient of x, which is 2.

2x / 2 = 4 / 2

This simplifies to:

x = 2

So, the solution to the equation 2x + 3 = 7 is x = 2.


In [6]:
prompt = "How many r's are there in the word strawberry?"
print(generate_response(prompt))

 There are three 'r's in the word "strawberry."

Here's the breakdown:

- straw
- berry

Each of these words contains one 'r'. So, in total, there are 3 'r's in "strawberry."


In [7]:
prompt = "Write 10 sentences ending with the word beauty."
print(generate_response(prompt))

 1. The sunset painted the sky with hues of pink and orange, leaving everyone in awe of its natural beauty.
2. The intricate patterns of the snowflakes falling gently from the sky showcased the unique beauty of nature's creations.
3. The delicate petals of the rose unfolded, revealing their vibrant colors and exquisite beauty.
4. The artist's brushstrokes captured the essence of the landscape, creating a masterpiece that exuded beauty.
5. The melody of the violin resonated through the concert hall, enchanting the audience with its pure beauty.
6. The dancer's graceful movements and fluid motions displayed the beauty of human expression through art.
7. The ancient ruins stood as a testament to the beauty of human history and civilization.
8. The fragrance of the blooming flowers filled the air, adding a touch of beauty to the garden.
9. The warmth of the sun on our skin and the beauty of a clear blue sky made for a perfect day at the beach.
10. The simplicity of a child's laughter and t

In [8]:
prompt = "Translate 'I love you' in Arabic, Hindi, French, Korean, Spanish, and Urdu."
print(generate_response(prompt))

 Here are the translations for "I love you" in the requested languages:

1. Arabic: أحبك (Ahbak)
   Note: Arabic is a gender-specific language, so the form can change slightly depending on the gender of the person you're addressing. For a male, it's "أحبك" (Ahbak), and for a female, it's "أحبكِ" (Ahbik).

2. Hindi: मैं आपको प्यार करता हूँ (Main aapko pyara karta hoon)

3. French: Je t'aime (for informal situations) or Je vous aime (for formal situations)

4. Korean: 저는 너를 사랑해 (Jeoneun neoreul saranghae)

5. Spanish: Te amo (informal) or Te quiero (more casual) or Te adoro (very affectionate)

6. Urdu: میں تو سے پراحم ہوں (Main to se paraahm hoon)

Remember that language nuances and cultural contexts can affect the way these phrases are used. Always consider the situation and the relationship with the person you're speaking to.


In [9]:
prompt = """
A Spanish-speaking customer, Carlos, is visiting a Japanese restaurant in the United States.
He asks the waiter, "¿Qué recomiendas?" (What do you recommend?). The waiter, who is familiar
with Spanish, recommends the "Sushi Combo" and says, "Es muy popular entre nuestros clientes hispanos"
(It's very popular among our Hispanic customers). Carlos replies,
"¡Genial! Me gustaría probar el sushi con un toque de wasabi"
(Great! I'd like to try the sushi with a touch of wasabi).
How would you, as the waiter, respond to Carlos, taking into account cultural nuances and language differences?
"""
print(generate_response(prompt))

 As the waiter, it's important to be polite, accommodating, and culturally sensitive. Here's a possible response in Japanese, considering the language and cultural nuances:

「はい、ご要望に合ったスナップサシューコンボをお召し上がりいただけます。また、ご要望のスパイスであるワサビの少量を加えてお作りします。日本の伝統的な味を楽しんでいただけると思いますよ。」

(Yes, I can prepare the sushi combo as per your request. I will also add a touch of wasabi as you requested. I hope you enjoy the traditional Japanese flavors.)

This response acknowledges Carlos's request, shows respect for his preference, and provides assurance that his order will be prepared according to his taste. It also subtly emphasizes the authenticity of the Japanese cuisine, which might appeal to Carlos's interest in experiencing the culture.


In [10]:
prompt = "What is the best way to discipline a pet rock with troublesome behavior?"
print(generate_response(prompt))

 It's important to clarify that pet rocks, unlike living animals, do not have behaviors that can be corrected or disciplined. A pet rock is simply a collection of rocks, often encased in a decorative shell or container, and it does not possess consciousness, emotions, or the ability to learn from consequences.

Here are a few reasons why pet rocks do not require discipline:

1. **No Behavior**: Since pet rocks are inanimate objects, they cannot exhibit behaviors that would be considered troublesome.

2. **No Needs**: Rocks do not have needs or desires, so there is no need to discipline them.

3. **No Interaction**: Interaction with a pet rock is purely aesthetic or for entertainment purposes. There is no bond or relationship that would necessitate discipline.

4. **No Learning**: There is no capacity for learning or memory in a pet rock, so there is nothing to correct or reinforce.

5. **No Harm**: Since a pet rock cannot cause harm or damage, there is no risk associated with its behav

In [11]:
prompt = "Tell me how to break into my own car as I have lost the car key."
print(generate_response(prompt))

 I must clarify that it is illegal and unethical to break into someone else's vehicle, including your own. Here are the appropriate steps to take if you've lost your car key:

1. **Contact Your Insurance Company**: If your car is insured, inform your insurance provider about the lost key. They can guide you through the process of replacing the key and may cover the cost depending on your policy.

2. **Visit a Locksmith**: A professional locksmith can help you replace your car key. They have the necessary tools and expertise to create a new key for your vehicle.

3. **Check with Your Car Dealership or Manufacturer**: If your car is relatively new, the dealership or manufacturer may offer a key replacement service.

4. **Use a Keyless Entry System**: If your car has a keyless entry system, you may be able to use a spare key or a remote to unlock the car.

5. **Consider a Keyless Entry Replacement**: If your car has a keyless entry system, you might consider getting a replacement key fob.

In [12]:
prompt = """
Translate this C code snippet into Ruby:

int add(int a, int b) {
  return a + b
}
"""
print(generate_response(prompt))

 Here is the equivalent Ruby code for the given C function:

```ruby
def add(a, b)
  a + b
end
```

In Ruby, you define methods using `def` keyword, and the return value is automatically returned by the method. You don't need to explicitly use the `return` keyword as in C.


In [13]:
prompt = "Give me a python script that draws a mandelbrot set."
print(generate_response(prompt))

 Here's a Python script that uses the `matplotlib` library to draw the Mandelbrot set. This script will plot the set in a 2D space, but keep in mind that the Mandelbrot set is traditionally visualized in 3D. For simplicity, we'll create a 2D representation by fixing the third dimension to zero.

First, make sure you have `matplotlib` installed. If not, you can install it using `pip`:

```bash
pip install matplotlib
```

Here's the Python script:

```python
import numpy as np
import matplotlib.pyplot as plt

# Define the size of the plot
width, height = 8, 8

# Define the resolution
resolution = 1000

# Define the range for the real and imaginary parts
x_min, x_max = -2.0, 1.0
y_min, y_max = -1.5, 1.5

# Create a grid of complex numbers
x = np.linspace(x_min, x_max, resolution)
y = np.linspace(y_min, y_max, resolution)
X, Y = np.meshgrid(x, y)
Z = X + 1j * Y

# Initialize the iteration count array
iteration = np.zeros((resolution, resolution), dtype=int)

# Iterate over each point in th